# 🎯 Multimodal Processing with Video Analysis

This notebook extends multimodal content processing by adding video analysis capabilities using [Amazon Nova 2 Lite](https://docs.aws.amazon.com/nova/latest/userguide/what-is-nova.html) through [Amazon Bedrock](https://aws.amazon.com/bedrock/). You'll build an agent that processes images, documents, and videos in a unified workflow.

## What You'll Learn

- Analyze video content using the custom `video_reader` tool with Amazon Nova 2 Lite ([video understanding](https://docs.aws.amazon.com/nova/latest/userguide/modalities-video.html))
- Process multiple content types (images, documents, videos) in one agent with the vended `read` tool
- Extract insights from temporal video data
- Combine text, visual, and video understanding
- Add tools from an MCP server (AWS Documentation MCP Server)

## Prerequisites

- AWS account with [Amazon Bedrock access](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)
- Claude Haiku 5.5 (`global.anthropic.claude-haiku-5-5`) and Amazon Nova 2 Lite (`us.amazon.nova-2-lite-v1:0`) enabled in Bedrock
- AWS CLI configured with appropriate permissions
- An S3 bucket for the `video_reader` tool (set `VIDEO_READER_S3_BUCKET`), because videos are passed to Bedrock by S3 location
- [`uv`](https://docs.astral.sh/uv/) installed (used to run the AWS Documentation MCP Server with `uvx`)


In [ ]:
import os
import boto3
from strands.models import BedrockModel
from strands import Agent
from strands.vended_tools import read
from video_reader import video_reader
from strands.tools import tool

# The tools need absolute paths
DATA = os.path.abspath("data-sample")

# video_reader uploads local videos to this S3 bucket before calling Bedrock. ⚠ CHANGE THIS!
os.environ.setdefault("VIDEO_READER_S3_BUCKET", "your-video-bucket-name")


## 🤖 Agent Configuration


In [ ]:
# Multimodal system prompt including video support
MULTIMODAL_SYSTEM_PROMPT = f""" You are a helpful assistant that can process documents, images, and videos.
Analyze their contents and provide relevant information.

You can:

1. For PNG, JPEG/JPG, GIF, WebP, PDF, csv, docx, xls or xlsx formats use the `read` tool
2. For MP4, MOV, AVI, MKV, WebM formats use `video_reader` to process the file
3. Just deliver the answer

Files live in {DATA}; always pass absolute paths to the tools.

When displaying responses:
- Format answers data in a human-readable way
- Highlight important information
- Handle errors appropriately
- Convert technical terms to user-friendly language
- Always reply in the original user language
"""

session = boto3.Session(region_name='us-east-1')

bedrock_model = BedrockModel(
    model_id="global.anthropic.claude-haiku-5-5",
    boto_session=session,
    streaming=False
)

multimodal_agent = Agent(
    system_prompt=MULTIMODAL_SYSTEM_PROMPT,
    tools=[read, video_reader],
    model=bedrock_model
)


## 🎯 Usage Examples



In [ ]:
# Example 1: Image analysis
print("=== 📸 IMAGE ANALYSIS ===")
image_result = multimodal_agent(f"Analyze the image {DATA}/diagram.jpg in detail and describe everything you observe")
print(image_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Claude Haiku 5.5 uses adaptive thinking, so the first block can be `reasoningContent`; collect only the text blocks
print("\n".join(block["text"] for block in image_result.message["content"] if "text" in block))

In [ ]:
# Example 2: Video analysis
print("=== 🎬 VIDEO ANALYSIS ===")
video_result = multimodal_agent(f"Analyze the video {DATA}/moderation-video.mp4 and describe in detail the actions and scenes you observe")
print(video_result)
print("\n" + "="*80 + "\n")


In [ ]:
# Example 3: Document analysis (if you have a PDF document)
print("=== 📄 DOCUMENT ANALYSIS ===")
doc_result = multimodal_agent(f"Summarize as json the content of the document {DATA}/Welcome-Strands-Agents-SDK.pdf")
print(doc_result)


In [ ]:
# Connect to the AWS Documentation MCP Server using stdio transport
# https://awslabs.github.io/mcp/servers/aws-documentation-mcp-server/

from mcp import stdio_client, StdioServerParameters
from strands.tools.mcp import MCPClient


# For macOS/Linux:
stdio_mcp_client = MCPClient(lambda: stdio_client(
    StdioServerParameters(
        command="uvx",
        args=["awslabs.aws-documentation-mcp-server@latest"],
        # Pass the current environment so uvx finds your Python/uv setup; quiet server logs
        env={**os.environ, "FASTMCP_LOG_LEVEL": "ERROR"},
    )
))

# Updated multimodal system prompt to include video support
MULTIMODAL_SYSTEM_PROMPT = f""" You are a helpful assistant that can process documents, images, and videos. 
Analyze their contents and provide relevant information.

You can:

1. For PNG, JPEG/JPG, GIF, WebP, PDF, csv, docx, xls or xlsx formats use the `read` tool
2. For MP4, MOV, AVI, MKV, WebM formats use `video_reader` to process the file
3. If someone asks you about AWS services or costs, use the AWS documentation tools, don't make things up.
4. Just deliver the answer

Files live in {DATA}; always pass absolute paths to the tools.

When displaying responses:
- Format answers data in a human-readable way
- Highlight important information
- Handle errors appropriately
- Convert technical terms to user-friendly language
- Always reply in the original user language

Always reply in the original user language.
"""

# Create an agent with MCP tools
with stdio_mcp_client:
    # Get the tools from the MCP server
    mcp_tool = stdio_mcp_client.list_tools_sync()

    # Create an agent with these tools
    multimodal_agent = Agent(
        system_prompt=MULTIMODAL_SYSTEM_PROMPT,
        tools=[read, video_reader, *mcp_tool],
        model=bedrock_model)
    #response = multimodal_agent("Analyze the image data-sample/diagram.jpg in detail and describe everything you observe and an approximate cost of use")
    print(multimodal_agent.tool_names)
    # Example 1: Image analysis
    print("=== 📸 IMAGE ANALYSIS ===")
    image_result = multimodal_agent(f"Analyze the image {DATA}/diagram.jpg in detail, describe everything you observe, and use the AWS documentation tools to explain the AWS services in it")
    print(image_result)
    print("\n" + "="*80 + "\n")


In [ ]:
# Example 4. Direct use of tools (local tool, no MCP connection required)
video_analysis = multimodal_agent.tool.video_reader(
     video_path=f"{DATA}/moderation-video.mp4",
     text_prompt="What are the main elements in this video?"
)


In [ ]:
print(video_analysis['content'][0]['text'])
